# 05 — DistilBERT amélioré : modèle hybride texte + métadonnées

## Motivation

Le DistilBERT standard (notebook 03) atteint **F1 macro ≈ 45%**, bien en dessous du Random Forest (57%).  
La raison principale : BERT n'a accès qu'au **texte**, alors que RF exploite `credibility_score` et `lie_rate` — les features les plus discriminantes du dataset.

## Approche : modèle hybride

```
Déclaration  →  DistilBERT  →  [CLS] (768 dim)
                                      ↓
Métadonnées  →  StandardScaler  →  [4 dim]
                                      ↓
                          concat (772 dim)
                                      ↓
                   Dropout → Linear(256) → GELU → Dropout → Linear(3)
                                      ↓
                          [fake, nuanced, real]
```

## Ajustements d'hyperparamètres vs baseline

| Paramètre | Baseline | Hybride | Raison |
|---|---|---|---|
| `learning_rate` | 2e-5 | 3e-5 | La tête custom doit apprendre plus vite |
| `num_epochs` | 3 | 4 | Une époque de plus pour que la tête converge |
| `warmup_ratio` | 0.10 | 0.06 | Warmup plus court avec lr plus élevé |

## 0. Imports

In [11]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import torch

from preprocessing import load_liar, preprocess_liar
from evaluate import evaluate_model
from bert_hybrid import (
    train_bert_hybrid,
    predict_bert_hybrid,
    HYBRID_META_COLS,
    fit_meta_scaler,
    get_meta_array,
)

print(f"PyTorch : {torch.__version__}")
print(f"MPS disponible : {torch.backends.mps.is_available()}")

PyTorch : 2.11.0
MPS disponible : True


## 1. Chargement des données

In [12]:
train_raw, valid_raw, test_raw = load_liar("data/raw")
train = preprocess_liar(train_raw)
valid = preprocess_liar(valid_raw)
test  = preprocess_liar(test_raw)

y_test = test["label_encoded"].values
print(f"Train : {len(train)} | Valid : {len(valid)} | Test : {len(test)}")
print(f"Métadonnées utilisées : {HYBRID_META_COLS}")

LIAR chargé — train: 10240, valid: 1284, test: 1267
Train : 10240 | Valid : 1284 | Test : 1267
Métadonnées utilisées : ['credibility_score', 'lie_rate', 'history_total', 'is_politician']


## 2. Résultats de référence

Pour situer l'objectif — ce que le modèle hybride doit dépasser.

In [13]:
baseline_results = pd.read_csv("outputs/results_indomain.csv")
print("Résultats existants (test set) :")
baseline_results[["model", "accuracy", "f1_macro", "f1_fake", "f1_nuanced", "f1_real"]]

Résultats existants (test set) :


,model,accuracy,f1_macro,f1_fake,f1_nuanced,f1_real
0,RF — test set,0.5651,0.5697,0.6431,0.4874,0.5786
1,XGBoost — test set,0.5556,0.5577,0.6239,0.4824,0.5668
2,LR — test set,0.5375,0.5442,0.6465,0.4705,0.5156
3,DistilBERT fine-tuné — test set,0.4672,0.4526,0.3667,0.4468,0.5444


## 3. Entraînement du modèle hybride

⏱ **Durée estimée : ~30-45 min sur Apple Silicon M4 Pro (MPS)**

Le modèle est sauvegardé dans `models/bert_hybrid/` à chaque époque — tu peux reprendre l'inférence sans réentraîner.

In [ ]:
trainer, model, scaler = train_bert_hybrid(
    train_df=train,
    valid_df=valid,
    model_name="distilbert-base-uncased",
    num_epochs=4,
    batch_size=32,
    learning_rate=3e-5,
    save_dir="models/bert_hybrid",
    max_length=128,
    meta_cols=HYBRID_META_COLS,
)

Device : Apple MPS

Chargement tokenizer : distilbert-base-uncased
Normalisation des métadonnées...
Tokenisation...

Initialisation DistilBertHybrid (backbone=distilbert-base-uncased, n_meta=4)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9331.04it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



Fine-tuning (4 époques, batch=32, lr=3e-05)...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.022713,1.007515,0.481308,0.483368,0.480501
2,0.954498,0.998762,0.496106,0.496330,0.494248
3,0.784273,1.067523,0.488318,0.484224,0.485641


## 4. Courbe d'apprentissage

Vérifie que la validation F1 s'améliore bien epoch après epoch et qu'il n'y a pas d'overfitting.

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history

train_loss = [(e["epoch"], e["loss"]) for e in log if "loss" in e and "eval_loss" not in e]
eval_f1    = [(e["epoch"], e["eval_f1_macro"]) for e in log if "eval_f1_macro" in e]
eval_loss  = [(e["epoch"], e["eval_loss"]) for e in log if "eval_loss" in e]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if train_loss:
    axes[0].plot(*zip(*train_loss), label="Train loss", color="#378ADD")
if eval_loss:
    axes[0].plot(*zip(*eval_loss), label="Valid loss", color="#D85A30", marker="o")
axes[0].set_title("Loss")
axes[0].set_xlabel("Époque")
axes[0].legend()
axes[0].grid(alpha=0.3)

if eval_f1:
    axes[1].plot(*zip(*eval_f1), color="#1a9850", marker="o", label="Valid F1 macro")
axes[1].set_title("F1 macro (validation)")
axes[1].set_xlabel("Époque")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("DistilBERT Hybride — courbes d'apprentissage", fontsize=13)
plt.tight_layout()
os.makedirs("outputs/figures", exist_ok=True)
plt.savefig("outputs/figures/learning_curve_bert_hybrid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée -> outputs/figures/learning_curve_bert_hybrid.png")

Figure sauvegardée -> outputs/figures/learning_curve_bert_hybrid.png


## 5. Évaluation sur le test set

In [ ]:
import torch.nn.functional as F

device = next(model.parameters()).device

# Recharge le meilleur checkpoint
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

from bert_hybrid import LIARHybridDataset
meta_test = get_meta_array(test, scaler, HYBRID_META_COLS)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

test_dataset = LIARHybridDataset(
    test["statement"].tolist(), meta_test, y_test_tensor, tokenizer
)

# Prédictions par batch
model.eval()
all_preds, all_probs = [], []
batch_size = 64

for i in range(0, len(test_dataset), batch_size):
    batch = [test_dataset[j] for j in range(i, min(i + batch_size, len(test_dataset)))]
    input_ids      = torch.stack([b["input_ids"] for b in batch]).to(device)
    attention_mask = torch.stack([b["attention_mask"] for b in batch]).to(device)
    meta_feat      = torch.stack([b["meta_features"] for b in batch]).to(device)

    with torch.no_grad():
        out   = model(input_ids=input_ids, attention_mask=attention_mask, meta_features=meta_feat)
        probs = F.softmax(out.logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

    all_preds.append(preds)
    all_probs.append(probs)

y_pred_hybrid = np.concatenate(all_preds)
y_prob_hybrid = np.vstack(all_probs)

In [ ]:
result_hybrid = evaluate_model(y_test, y_pred_hybrid, "DistilBERT Hybride — test set")


  DistilBERT Hybride — test set
  Accuracy    : 0.5083
  F1 macro    : 0.5025   ← métrique principale
  F1 weighted : 0.5030

  F1 par classe :
    fake       : 0.4832
    nuanced    : 0.4519
    real       : 0.5723

              precision    recall  f1-score   support

        fake       0.50      0.46      0.48       341
     nuanced       0.51      0.41      0.45       477
        real       0.51      0.65      0.57       449

    accuracy                           0.51      1267
   macro avg       0.51      0.51      0.50      1267
weighted avg       0.51      0.51      0.50      1267



## 6. Comparaison globale

In [ ]:
df_new = pd.DataFrame([result_hybrid])
df_all = pd.concat([baseline_results, df_new], ignore_index=True)
df_all = df_all.sort_values("f1_macro", ascending=False).reset_index(drop=True)
df_all[["model", "accuracy", "f1_macro", "f1_fake", "f1_nuanced", "f1_real"]]

,model,accuracy,f1_macro,f1_fake,f1_nuanced,f1_real
0,RF — test set,0.5651,0.5697,0.6431,0.4874,0.5786
1,XGBoost — test set,0.5556,0.5577,0.6239,0.4824,0.5668
2,LR — test set,0.5375,0.5442,0.6465,0.4705,0.5156
3,DistilBERT Hybride — test set,0.5083,0.5025,0.4832,0.4519,0.5723
4,DistilBERT fine-tuné — test set,0.4672,0.4526,0.3667,0.4468,0.5444


In [ ]:
# Visualisation comparative
import matplotlib.pyplot as plt
import numpy as np

models  = df_all["model"].tolist()
f1_vals = df_all["f1_macro"].tolist()
colors  = ["#1a9850" if "Hybride" in m else "#378ADD" if "RF" in m or "XGBoost" in m or "LR" in m else "#D85A30"
           for m in models]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(models[::-1], f1_vals[::-1], color=colors[::-1], alpha=0.85)

for bar, val in zip(bars, f1_vals[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=9)

ax.set_xlabel("F1 macro")
ax.set_title("Comparaison tous modèles — test set LIAR", fontsize=13)
ax.set_xlim(0, 0.75)
ax.axvline(x=df_all["f1_macro"].max(), color="gray", linestyle="--", alpha=0.4, label="Meilleur modèle")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/figures/comparison_all_models.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée -> outputs/figures/comparison_all_models.png")

Figure sauvegardée -> outputs/figures/comparison_all_models.png


In [ ]:
# Sauvegarde des résultats complets
df_all.to_csv("outputs/results_all_models.csv", index=False)
print("Résultats sauvegardés -> outputs/results_all_models.csv")

# Delta vs DistilBERT baseline
bert_base = baseline_results[baseline_results["model"].str.contains("DistilBERT")]["f1_macro"].values
if len(bert_base):
    delta = result_hybrid["f1_macro"] - bert_base[0]
    print(f"\nGain DistilBERT Hybride vs DistilBERT standard : {delta*100:+.2f} pp")

rf_f1 = baseline_results[baseline_results["model"].str.contains("RF")]["f1_macro"].values
if len(rf_f1):
    delta_rf = result_hybrid["f1_macro"] - rf_f1[0]
    print(f"Gain DistilBERT Hybride vs Random Forest      : {delta_rf*100:+.2f} pp")

Résultats sauvegardés -> outputs/results_all_models.csv

Gain DistilBERT Hybride vs DistilBERT standard : +4.99 pp
Gain DistilBERT Hybride vs Random Forest      : -6.72 pp
